# Retrieval-augmented generation

Read the [guide](README.md) for prerequisites, costs and exercises. Run all cells in order with the `notebooks`, `rag`, `memory` and `aws` groups installed. Calls use offline fixtures, not live model judgments.


In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_load_and_chunk

[Script](examples/01_load_and_chunk.py). Predict the result before running.


In [ ]:
"""Load approved Markdown and PDF policies with stable source/page citations.

Run: uv run python modules/07_rag/examples/01_load_and_chunk.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli
from common.retrieval import load_chunks, split_text


def main() -> None:
    args, cfg = lesson_settings()
    try:
        split_text("A chunk must advance", size=3, overlap=3)
    except ValueError as exc:
        console.print("Failure before fix:", str(exc))
    # KEY IDEA: never concatenate PDF pages before assigning provenance.
    for chunk in load_chunks(pdf=True)[:3]:
        console.print(chunk.record())
    console.print("Markdown chunks:", len(load_chunks()))
    complete()


run_lesson(main, "modules/07_rag/examples/01_load_and_chunk.py")

## 02_embeddings_by_hand

[Script](examples/02_embeddings_by_hand.py). Predict the result before running.


In [ ]:
"""Compute cosine similarity and plot a deliberately limited 2D teaching projection.

Run: uv run --group rag python modules/07_rag/examples/02_embeddings_by_hand.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import matplotlib

from common.config import ROOT
from common.retrieval import cosine

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    vectors = {"refund timing": [3, 1], "refund approval": [2, 1], "aisle seat": [0, 3]}
    console.print("Dot=7; norms=sqrt(10),sqrt(5); cosine:", cosine([3, 1], [2, 1]))
    # KEY IDEA: directions can be similar even when vector lengths differ.
    fig, ax = plt.subplots(figsize=(7, 4))
    for label, (x, y) in vectors.items():
        ax.scatter(x, y, s=60)
        ax.annotate(label, (x, y), xytext=(8, 8), textcoords="offset points")
    ax.set(
        xlim=(-0.3, 4.1),
        ylim=(0, 3.7),
        xlabel="Illustrative refund dimension",
        ylabel="Illustrative preference dimension",
        title="Toy vectors, not learned model embeddings",
    )
    ax.grid(alpha=0.2)
    fig.tight_layout()
    output = ROOT / ".nimbus/embedding_geometry.png"
    output.parent.mkdir(exist_ok=True)
    fig.savefig(output, dpi=160)
    plt.close(fig)
    console.print("Plot:", str(output))
    complete()


run_lesson(main, "modules/07_rag/examples/02_embeddings_by_hand.py")

## 03_local_index

[Script](examples/03_local_index.py). Predict the result before running.


In [ ]:
"""Persist explicit Ollama vectors in Chroma; reopen without re-embedding the corpus.

Run: uv run --group rag python modules/07_rag/examples/03_local_index.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.config import ROOT
from common.lessons import complete, lesson_settings
from common.models import get_embeddings
from common.pretty import console, run_cli
from common.retrieval import LocalIndex, load_chunks


def main() -> None:
    args, cfg = lesson_settings()
    embed = get_embeddings(mock=args.mock)
    directory = ROOT / ".nimbus" / ("chroma_mock" if args.mock else "chroma_ollama")
    index = LocalIndex(directory, embed)
    index.replace(load_chunks())
    # KEY IDEA: the same embedding identity must serve indexing and querying.
    reopened = LocalIndex(directory, embed)
    console.print(reopened.search("approved refund timing", topic="refund"))
    complete()


run_lesson(main, "modules/07_rag/examples/03_local_index.py")

## 04_cited_search

[Script](examples/04_cited_search.py). Predict the result before running.


In [ ]:
"""Let the agent search, then validate its citations against retrieved chunks.

Run: uv run python modules/07_rag/examples/04_cited_search.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli
from common.rag_agent import answer_policy
from common.retrieval import load_chunks


def main() -> None:
    args, cfg = lesson_settings()
    # KEY IDEA: source/quote validation is application code, not a promise in the prompt.
    result, evidence = answer_policy(cfg, "approved refund timing", load_chunks(pdf=True))
    console.print(result.structured_output.model_dump())
    console.print("Retrieved chunks:", len(evidence))
    complete(result, cfg)


run_lesson(main, "modules/07_rag/examples/04_cited_search.py")

## 05_knowledge_base

[Script](examples/05_knowledge_base.py). Predict the result before running.


In [ ]:
"""AWS optional: supported memory-store retrieval with an exact metadata filter.

Run: uv run python modules/07_rag/examples/05_knowledge_base.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

import asyncio

from common.aws_retrieval import knowledge_store
from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli


def main() -> None:
    args, cfg = lesson_settings()
    with knowledge_store(args.mock) as store:
        # KEY IDEA: a read-only store replaces the deprecated retrieve community tool.
        entries = asyncio.run(store.search("refund timing"))
        console.print(entries)
    complete()


run_lesson(main, "modules/07_rag/examples/05_knowledge_base.py")

## 06_bedrock_embeddings

[Script](examples/06_bedrock_embeddings.py). Predict the result before running.


In [ ]:
"""AWS optional: Titan embeddings use InvokeModel because Converse does not embed.

Run: uv run python modules/07_rag/examples/06_bedrock_embeddings.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.aws_retrieval import titan_vector
from common.lessons import complete, lesson_settings
from common.pretty import console, run_cli
from common.retrieval import cosine


def main() -> None:
    args, cfg = lesson_settings()
    # KEY IDEA: embedding inference has a different API from chat inference.
    vector = titan_vector("Nimbus refund policy", mock=args.mock)
    console.print("Dimensions:", len(vector), "self cosine:", cosine(vector, vector))
    complete()


run_lesson(main, "modules/07_rag/examples/06_bedrock_embeddings.py")

## ask_nimbus

[Script](use_case/ask_nimbus.py). Predict the result before running.


In [ ]:
"""Ask Nimbus: persistent vector retrieval and evidence-checked policy answers.

Run: uv run --group rag python modules/07_rag/use_case/ask_nimbus.py --mock
Expected: Lesson complete.
Cost: offline $0; live model/embedding calls use configured rates.
AWS resources created by this script: none. Optional labs have separate CDK/cleanup.
"""

from common.config import ROOT
from common.lessons import complete, lesson_settings
from common.models import get_embeddings
from common.pretty import console, run_cli
from common.rag_agent import answer_policy
from common.retrieval import LocalIndex, load_chunks


def main() -> None:
    args, cfg = lesson_settings()
    chunks = load_chunks(pdf=True)
    by_id = {c.id: c for c in chunks}
    index = LocalIndex(
        ROOT / ".nimbus" / ("ask_nimbus_mock" if args.mock else "ask_nimbus_live"),
        get_embeddings(mock=args.mock),
    )
    index.replace(chunks)
    # KEY IDEA: retrieve IDs map to authoritative stored chunks, never model-invented filenames.
    result, _ = answer_policy(
        cfg,
        "How long does an approved refund take?",
        chunks,
        retrieve=lambda query: [by_id[row["id"]] for row in index.search(query)],
    )
    console.print(result.structured_output.model_dump())
    complete(result, cfg)


run_lesson(main, "modules/07_rag/use_case/ask_nimbus.py")

## Practice

Complete the [exercises](README.md#exercises) and [quiz](quiz.md), then run the module tests.
